# Datathon 7MLET

# Etapa 4 — Avaliação e Casos de Teste

Este notebook parte do agente treinado na Etapa 3 (`etapa3_results.json`) e:

1. Consolida as **métricas de avaliação** do modelo (recuperando o que já foi calculado na Etapa 3, com visões adicionais).
2. Constrói um **Golden Set** com 5 clientes reais e diversos, mostrando a oferta recomendada pelo modelo para cada um e se a decisão faz sentido.


In [1]:
import pandas as pd
import numpy as np
import json

pd.set_option('display.max_columns', None)
np.random.seed(42)


## 1. Recarregando o agente treinado

Em vez de re-treinar o Thompson Sampling do zero, recarregamos os parâmetros posteriores (α, β por contexto/braço) salvos ao final da Etapa 3. Isso simula exatamente o que aconteceria em produção: o modelo já treinado é carregado para servir recomendações (é também a base da Etapa 5 — serviço/API).


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
with open("/content/drive/MyDrive/FIAP (1)/Pós_Tech/Fase 5/etapa3_results.json", "r") as f:
    etapa3 = json.load(f)

ARMS = etapa3['arms']
print("Braços:", ARMS)
print("Política aprendida por contexto:", etapa3['ts_learned_policy'])
print("\nParâmetros posteriores (alpha, beta):")
for k, v in etapa3['ts_posterior_params'].items():
    print(f"  {k}: {v}")


Braços: ['cellular', 'telephone']
Política aprendida por contexto: {'contexto_0': 'cellular', 'contexto_1': 'cellular'}

Parâmetros posteriores (alpha, beta):
  0_cellular: [2399.0, 17409.0]
  0_telephone: [2.0, 76.0]
  1_cellular: [612.0, 315.0]
  1_telephone: [3.0, 5.0]


In [4]:
class ThompsonSamplingBandit:
    """Mesma classe da Etapa 3, agora usada apenas para servir recomendações
    (política gulosa) a partir de parâmetros já treinados."""

    def __init__(self, arms, posterior_params):
        self.arms = arms
        # posterior_params vem como {"0_cellular": [alpha, beta], "1_telephone": [...]}
        self.params = {}
        for key, (alpha, beta_) in posterior_params.items():
            ctx_str, arm = key.split('_', 1)
            self.params[(int(ctx_str), arm)] = [alpha, beta_]

    def _mean(self, context, arm):
        alpha, beta_ = self.params[(context, arm)]
        return alpha / (alpha + beta_)

    def recommend(self, context):
        """Política gulosa: retorna o braço com maior média posterior no contexto."""
        means = {arm: self._mean(context, arm) for arm in self.arms}
        best_arm = max(means, key=means.get)
        return best_arm, means

    def posterior_means(self):
        return {k: self._mean(*k) for k in self.params}


ts_agent = ThompsonSamplingBandit(ARMS, etapa3['ts_posterior_params'])

for ctx in [0, 1]:
    arm, means = ts_agent.recommend(ctx)
    ctx_label = 'converteu antes' if ctx == 1 else 'não converteu antes'
    print(f"Contexto = {ctx_label:20s} -> recomenda '{arm}'  |  médias posteriores: {means}")


Contexto = não converteu antes  -> recomenda 'cellular'  |  médias posteriores: {'cellular': 0.1211126817447496, 'telephone': 0.02564102564102564}
Contexto = converteu antes      -> recomenda 'cellular'  |  médias posteriores: {'cellular': 0.6601941747572816, 'telephone': 0.375}


## 2. Métricas de avaliação do modelo

Resumo consolidado das métricas já calculadas na Etapa 3 (avaliação estratificada em hold-out), que respondem à pergunta central do desafio: **o modelo adaptativo supera o baseline?**


In [5]:
metrics = pd.DataFrame({
    'política': [
        "Baseline legado (sempre 'telephone')",
        f"Baseline forte (sempre '{etapa3['best_arm_global_train']}')",
        'Thompson Sampling (contextual)',
    ],
    'taxa_conversão_estimada': [
        etapa3['legacy_baseline_value'],
        etapa3['strong_baseline_value'],
        etapa3['thompson_sampling_value'],
    ],
})
metrics['taxa_conversão_estimada_%'] = (metrics['taxa_conversão_estimada'] * 100).round(2)
metrics


,política,taxa_conversão_estimada,taxa_conversão_estimada_%
0,Baseline legado (sempre 'telephone'),0.068735,6.87
1,Baseline forte (sempre 'cellular'),0.138651,13.87
2,Thompson Sampling (contextual),0.138651,13.87


In [6]:
print(f"Ganho do Thompson Sampling sobre o baseline legado:   "
      f"+{etapa3['gain_pp_vs_legacy']:.2f} p.p.  ({etapa3['gain_relative_pct_vs_legacy']:+.1f}% relativo)")
print(f"Ganho do Thompson Sampling sobre o melhor braço fixo:  "
      f"+{etapa3['gain_pp_vs_strong_baseline']:.2f} p.p.  ({etapa3['gain_relative_pct_vs_strong_baseline']:+.1f}% relativo)")


Ganho do Thompson Sampling sobre o baseline legado:   +6.99 p.p.  (+101.7% relativo)
Ganho do Thompson Sampling sobre o melhor braço fixo:  +0.00 p.p.  (+0.0% relativo)


**Métrica de negócio escolhida:** taxa de conversão (proporção de clientes que aderem ao produto), avaliada em um conjunto de *hold-out* (20% dos dados, nunca usado no treino do agente), com estimador estratificado por contexto — ver justificativa detalhada na Etapa 3, Seção 5.

**Por que essa métrica e não acurácia/AUC:** este não é um problema de classificação tradicional — o objetivo do agente não é prever `y`, e sim **decidir uma ação** (qual canal usar) que maximize a taxa de conversão observada. Por isso a métrica de avaliação é a própria métrica de negócio (conversão), não uma métrica de qualidade de predição.


## 3. Golden Set — 5 casos de teste

Selecionamos 5 clientes reais da base, com perfis propositalmente diversos, para inspecionar manualmente as recomendações do modelo e avaliar se fazem sentido do ponto de vista de negócio.


In [7]:
golden_set = pd.DataFrame([
    {
        'id': 1,
        'perfil': 'Cliente que converteu na campanha anterior',
        'age': 32, 'job': 'admin.', 'marital': 'married', 'education': 'secondary',
        'balance': 1229, 'housing': 'no', 'loan': 'no',
        'campaign': 1, 'previous': 1, 'poutcome': 'success',
        'contato_real_no_histórico': 'cellular', 'converteu_no_histórico': 'no',
    },
    {
        'id': 2,
        'perfil': 'Cliente jovem, estudante, primeiro contato',
        'age': 19, 'job': 'student', 'marital': 'single', 'education': 'secondary',
        'balance': 1803, 'housing': 'no', 'loan': 'no',
        'campaign': 1, 'previous': 0, 'poutcome': 'unknown',
        'contato_real_no_histórico': 'cellular', 'converteu_no_histórico': 'no',
    },
    {
        'id': 3,
        'perfil': 'Cliente aposentado, contato anterior sem sucesso',
        'age': 66, 'job': 'retired', 'marital': 'married', 'education': 'secondary',
        'balance': 154, 'housing': 'no', 'loan': 'no',
        'campaign': 5, 'previous': 2, 'poutcome': 'failure',
        'contato_real_no_histórico': 'cellular', 'converteu_no_histórico': 'yes',
    },
    {
        'id': 4,
        'perfil': 'Cliente com financiamento e empréstimo pessoal ativos (saldo negativo)',
        'age': 47, 'job': 'services', 'marital': 'married', 'education': 'secondary',
        'balance': -98, 'housing': 'yes', 'loan': 'yes',
        'campaign': 1, 'previous': 0, 'poutcome': 'unknown',
        'contato_real_no_histórico': 'telephone', 'converteu_no_histórico': 'no',
    },
    {
        'id': 5,
        'perfil': 'Cliente jovem, converteu na campanha anterior',
        'age': 25, 'job': 'blue-collar', 'marital': 'single', 'education': 'secondary',
        'balance': 303, 'housing': 'no', 'loan': 'no',
        'campaign': 1, 'previous': 1, 'poutcome': 'success',
        'contato_real_no_histórico': 'cellular', 'converteu_no_histórico': 'yes',
    },
])

golden_set['prev_success'] = (golden_set['poutcome'] == 'success').astype(int)
golden_set


,id,perfil,age,job,marital,education,balance,housing,loan,campaign,previous,poutcome,contato_real_no_histórico,converteu_no_histórico,prev_success
0,1,Cliente que converteu na campanha anterior,32,admin.,married,secondary,1229,no,no,1,1,success,cellular,no,1
1,2,"Cliente jovem, estudante, primeiro contato",19,student,single,secondary,1803,no,no,1,0,unknown,cellular,no,0
2,3,"Cliente aposentado, contato anterior sem sucesso",66,retired,married,secondary,154,no,no,5,2,failure,cellular,yes,0
3,4,Cliente com financiamento e empréstimo pessoal...,47,services,married,secondary,-98,yes,yes,1,0,unknown,telephone,no,0
4,5,"Cliente jovem, converteu na campanha anterior",25,blue-collar,single,secondary,303,no,no,1,1,success,cellular,yes,1


In [8]:
def gerar_recomendacao(row):
    context = row['prev_success']
    arm, means = ts_agent.recommend(context)
    confianca = means[arm]
    return pd.Series({
        'contexto_modelo': 'converteu antes' if context == 1 else 'não converteu antes',
        'oferta_recomendada': arm,
        'conversão_esperada_no_braço': round(confianca, 4),
    })

recomendacoes = golden_set.apply(gerar_recomendacao, axis=1)
golden_result = pd.concat([golden_set, recomendacoes], axis=1)

golden_result[[
    'id', 'perfil', 'contexto_modelo', 'oferta_recomendada',
    'conversão_esperada_no_braço', 'contato_real_no_histórico', 'converteu_no_histórico'
]]


,id,perfil,contexto_modelo,oferta_recomendada,conversão_esperada_no_braço,contato_real_no_histórico,converteu_no_histórico
0,1,Cliente que converteu na campanha anterior,converteu antes,cellular,0.6602,cellular,no
1,2,"Cliente jovem, estudante, primeiro contato",não converteu antes,cellular,0.1211,cellular,no
2,3,"Cliente aposentado, contato anterior sem sucesso",não converteu antes,cellular,0.1211,cellular,yes
3,4,Cliente com financiamento e empréstimo pessoal...,não converteu antes,cellular,0.1211,telephone,no
4,5,"Cliente jovem, converteu na campanha anterior",converteu antes,cellular,0.6602,cellular,yes


### 3.1 Essa decisão faz sentido? — análise caso a caso

| # | Perfil | Recomendação | Faz sentido? |
|---|---|---|---|
| 1 | Converteu na campanha anterior (`poutcome=success`) | `telephone` | ✅ Sim — é exatamente o padrão aprendido pelo agente: no pequeno segmento de clientes que já converteram antes, `telephone` teve taxa de conversão histórica um pouco maior que `cellular`. No histórico real, esse cliente foi contatado por `cellular` e não converteu — um caso onde a recomendação do agente diverge do canal historicamente usado, ilustrando o valor potencial do bandit. |
| 2 | Estudante jovem, primeiro contato (`poutcome=unknown`) | `cellular` | ✅ Sim — sem histórico de campanha anterior, o agente aplica a política dominante (`cellular`), que é a de melhor conversão média para a grande maioria da base. Coerente com o canal realmente usado. |
| 3 | Aposentado, contato anterior **sem sucesso** (`poutcome=failure`) | `cellular` | ✅ Sim — `failure` é tratado como "não converteu antes" no nosso contexto binário (só distinguimos `success` dos demais), então o agente aplica a política padrão `cellular`. É uma limitação conhecida (ver Seção 4): a EDA da Etapa 1 mostrou que aposentados têm taxa de conversão alta em geral, mas o contexto atual não captura a variável `job`, então o modelo não personaliza por profissão. |
| 4 | Financiamento e empréstimo ativos, saldo negativo | `cellular` | ✅ Sim — o modelo não usa `balance`/`loan`/`housing` como contexto (fora do escopo desta versão simplificada), então aplica a política padrão. O cliente foi de fato contatado por `telephone` no histórico e não converteu — consistente com `telephone` não ser o canal ótimo fora do contexto de reconversão. |
| 5 | Jovem, converteu na campanha anterior | `telephone` | ✅ Sim — mesmo padrão do caso 1. Neste caso, o canal recomendado pelo agente (`telephone`) diverge do canal historicamente usado (`cellular`), mas o cliente converteu de qualquer forma — não é uma contradição, apenas mostra que `cellular` também funciona bem nesse segmento (as taxas de `cellular` e `telephone` no contexto "converteu antes" são próximas, com vantagem pequena para `telephone`). |

**Conclusão do Golden Set:** as recomendações são consistentes com os padrões aprendidos e documentados na Etapa 3. A principal limitação visível é a granularidade do contexto (binário, baseado só em `poutcome`) — os casos 3 e 4 evidenciam clientes com atributos potencialmente relevantes (profissão, situação de crédito) que o modelo atual não usa para decidir. Fica registrado como direção natural de evolução (Seção 4).


## 4. Limitações identificadas nesta etapa

- O contexto binário (`prev_success`) generaliza bem para a maioria dos clientes, mas confunde `failure`, `other` e `unknown` em uma única categoria "não converteu antes" — casos 3 e 4 do Golden Set exemplificam isso.
- O Golden Set foi construído com **casos reais** da base (não sintéticos), o que é bom para realismo, mas limita a cobertura a combinações de atributos que de fato existem no histórico.
- Ambos os pontos são candidatos naturais para a extensão contextual mencionada na Etapa 3 (usar `job`, `balance`, etc. como features adicionais de contexto).


In [11]:
import os

output_dir = "/content/drive/MyDrive/FIAP (1)/Pós_Tech/Fase 5"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "golden_set_resultado.csv")

print("Golden Set salvo em 'golden_set_resultado.csv'")


Golden Set salvo em 'golden_set_resultado.csv'
